# Exploring the DuckDB Warehouse

This notebook lets you inspect the database structure and data behind the Text-to-SQL analytics agent.

**What's in the database?**
- `raw_daily_prices` — Raw OHLCV data pulled from yfinance
- `fact_daily_prices` — Cleaned prices + computed daily returns
- `dim_securities` — Ticker metadata (sector, industry, exchange, etc.)
- `dim_calendar` — Trading calendar with date attributes per exchange

## 1. Connect to DuckDB

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("data/warehouse.duckdb", read_only=True)
print("Connected to DuckDB warehouse")

Connected to DuckDB warehouse


## 2. What tables exist?

In [2]:
con.execute("SHOW TABLES").fetchdf()

,name
0,dim_calendar
1,dim_securities
2,fact_daily_prices
3,raw_daily_prices


## 3. Table schemas (columns & types)

In [3]:
tables = con.execute("SHOW TABLES").fetchdf()["name"].tolist()

for table in tables:
    print(f"{'='*60}")
    print(f"  {table}")
    print(f"{'='*60}")
    schema = con.execute(f"DESCRIBE {table}").fetchdf()
    display(schema[["column_name", "column_type", "null", "key"]])
    
    row_count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  → {row_count:,} rows\n")

  dim_calendar


,column_name,column_type,null,key
0,date,DATE,NO,PRI
1,exchange,VARCHAR,NO,PRI
2,year,INTEGER,YES,NaN
3,quarter,INTEGER,YES,NaN
4,month,INTEGER,YES,NaN
5,day_of_week,INTEGER,YES,NaN
6,day_name,VARCHAR,YES,NaN
7,is_month_start,BOOLEAN,YES,NaN
8,is_month_end,BOOLEAN,YES,NaN
9,is_quarter_start,BOOLEAN,YES,NaN


  → 3,864 rows

  dim_securities


,column_name,column_type,null,key
0,ticker,VARCHAR,NO,PRI
1,short_name,VARCHAR,YES,NaN
2,long_name,VARCHAR,YES,NaN
3,security_type,VARCHAR,NO,NaN
4,sector,VARCHAR,YES,NaN
5,industry,VARCHAR,YES,NaN
6,exchange,VARCHAR,YES,NaN
7,currency,VARCHAR,YES,NaN
8,market_cap,BIGINT,YES,NaN
9,country,VARCHAR,YES,NaN


  → 24 rows

  fact_daily_prices


,column_name,column_type,null,key
0,ticker,VARCHAR,NO,PRI
1,date,DATE,NO,PRI
2,open,DOUBLE,YES,NaN
3,high,DOUBLE,YES,NaN
4,low,DOUBLE,YES,NaN
5,close,DOUBLE,YES,NaN
6,adjusted_close,DOUBLE,YES,NaN
7,volume,BIGINT,YES,NaN
8,daily_simple_return,DOUBLE,YES,NaN
9,daily_log_return,DOUBLE,YES,NaN


  → 30,912 rows

  raw_daily_prices


,column_name,column_type,null,key
0,ticker,VARCHAR,NO,PRI
1,date,DATE,NO,PRI
2,open,DOUBLE,YES,NaN
3,high,DOUBLE,YES,NaN
4,low,DOUBLE,YES,NaN
5,close,DOUBLE,YES,NaN
6,adjusted_close,DOUBLE,YES,NaN
7,volume,BIGINT,YES,NaN


  → 30,912 rows



## 4. Sample data from each table

In [4]:
con.execute("SELECT * FROM fact_daily_prices LIMIT 5").fetchdf()

,ticker,date,open,high,low,close,adjusted_close,volume,daily_simple_return,daily_log_return
0,AAPL,2021-06-17,129.800003,132.550003,129.649994,131.789993,128.462555,96721700,NaN,NaN
1,AAPL,2021-06-18,130.710007,131.509995,130.240005,130.460007,127.166138,108953300,-0.010092,-0.010143
2,AAPL,2021-06-21,130.300003,132.410004,129.210007,132.300003,128.959686,79663300,0.014104,0.014005
3,AAPL,2021-06-22,132.130005,134.080002,131.619995,133.979996,130.597275,74783600,0.012698,0.012619
4,AAPL,2021-06-23,133.770004,134.320007,133.229996,133.699997,130.324341,60214200,-0.002090,-0.002092


In [5]:
con.execute("SELECT * FROM dim_securities").fetchdf()

,ticker,short_name,long_name,security_type,sector,industry,exchange,currency,market_cap,country
0,AAPL,Apple Inc.,Apple Inc.,equity,Technology,Consumer Electronics,NMS,USD,4376391581696,United States
1,MSFT,Microsoft Corporation,Microsoft Corporation,equity,Technology,Software - Infrastructure,NMS,USD,2923258445824,United States
2,GOOGL,Alphabet Inc.,Alphabet Inc.,equity,Communication Services,Internet Content & Information,NMS,USD,4572181954560,United States
3,AMZN,"Amazon.com, Inc.","Amazon.com, Inc.",equity,Consumer Cyclical,Internet Retail,NMS,USD,2682285457408,United States
4,NVDA,NVIDIA Corporation,NVIDIA Corporation,equity,Technology,Semiconductors,NMS,USD,5088589905920,United States
5,META,"Meta Platforms, Inc.","Meta Platforms, Inc.",equity,Communication Services,Internet Content & Information,NMS,USD,1516682477568,United States
6,TSLA,"Tesla, Inc.","Tesla, Inc.",equity,Consumer Cyclical,Auto Manufacturers,NMS,USD,1534025400320,United States
7,JPM,JP Morgan Chase & Co.,JPMorgan Chase & Co.,equity,Financial Services,Banks - Diversified,NYQ,USD,881425252352,United States
8,V,Visa Inc.,Visa Inc.,equity,Financial Services,Credit Services,NYQ,USD,626244190208,United States
9,JNJ,Johnson & Johnson,Johnson & Johnson,equity,Healthcare,Drug Manufacturers - General,NYQ,USD,559738126336,United States


In [ ]:
con.execute("SELECT * FROM dim_calendar LIMIT 10").fetchdf()

## 5. Quick stats — date range, tickers, coverage

In [6]:
con.execute("""
    SELECT
        COUNT(DISTINCT ticker) AS num_tickers,
        MIN(date) AS earliest_date,
        MAX(date) AS latest_date,
        COUNT(*) AS total_rows
    FROM fact_daily_prices
""").fetchdf()

,num_tickers,earliest_date,latest_date,total_rows
0,24,2021-06-17,2026-06-15,30912


In [7]:
# Rows per ticker — useful to spot missing data
con.execute("""
    SELECT
        f.ticker,
        s.short_name,
        s.security_type,
        s.sector,
        COUNT(*) AS trading_days,
        MIN(f.date) AS first_date,
        MAX(f.date) AS last_date
    FROM fact_daily_prices f
    JOIN dim_securities s ON f.ticker = s.ticker
    GROUP BY f.ticker, s.short_name, s.security_type, s.sector
    ORDER BY trading_days DESC
""").fetchdf()

,ticker,short_name,security_type,sector,trading_days,first_date,last_date
0,ORK.OL,ORKLA,equity,Consumer Defensive,1288,2021-06-17,2026-06-15
1,AAPL,Apple Inc.,equity,Technology,1288,2021-06-17,2026-06-15
2,PFE,"Pfizer, Inc.",equity,Healthcare,1288,2021-06-17,2026-06-15
3,HD,"Home Depot, Inc. (The)",equity,Consumer Cyclical,1288,2021-06-17,2026-06-15
4,META,"Meta Platforms, Inc.",equity,Communication Services,1288,2021-06-17,2026-06-15
5,NVDA,NVIDIA Corporation,equity,Technology,1288,2021-06-17,2026-06-15
6,DNB.OL,DNB BANK ASA,equity,Financial Services,1288,2021-06-17,2026-06-15
7,MOWI.OL,MOWI,equity,Consumer Defensive,1288,2021-06-17,2026-06-15
8,UNH,UnitedHealth Group Incorporated,equity,Healthcare,1288,2021-06-17,2026-06-15
9,TEL.OL,TELENOR,equity,Communication Services,1288,2021-06-17,2026-06-15


## 6. How tables relate (joins)

```
dim_securities.ticker  ──┐
                         ├──  fact_daily_prices.ticker
dim_calendar.date  ──────┤
                         └──  fact_daily_prices.date

dim_calendar is built from raw_daily_prices × dim_securities (equity tickers only)
```

Try joining them yourself:

In [8]:
# Example join: latest closing price + sector for each ticker
con.execute("""
    SELECT
        s.ticker,
        s.short_name,
        s.sector,
        s.exchange,
        f.date AS latest_date,
        f.close AS latest_close,
        f.daily_simple_return
    FROM fact_daily_prices f
    JOIN dim_securities s ON f.ticker = s.ticker
    WHERE f.date = (SELECT MAX(date) FROM fact_daily_prices WHERE ticker = f.ticker)
    ORDER BY s.sector, s.ticker
""").fetchdf()

,ticker,short_name,sector,exchange,latest_date,latest_close,daily_simple_return
0,GOOGL,Alphabet Inc.,Communication Services,NMS,2026-06-15,NaN,NaN
1,META,"Meta Platforms, Inc.",Communication Services,NMS,2026-06-15,593.479980,0.047709
2,TEL.OL,TELENOR,Communication Services,OSE,2026-06-15,NaN,NaN
3,AMZN,"Amazon.com, Inc.",Consumer Cyclical,NMS,2026-06-15,NaN,NaN
4,HD,"Home Depot, Inc. (The)",Consumer Cyclical,NYQ,2026-06-15,NaN,NaN
5,TSLA,"Tesla, Inc.",Consumer Cyclical,NMS,2026-06-15,NaN,NaN
6,MOWI.OL,MOWI,Consumer Defensive,OSE,2026-06-15,NaN,NaN
7,ORK.OL,ORKLA,Consumer Defensive,OSE,2026-06-15,NaN,NaN
8,PG,Procter & Gamble Company (The),Consumer Defensive,NYQ,2026-06-15,NaN,NaN
9,AKRBP.OL,AKER BP,Energy,OSE,2026-06-15,321.299988,-0.064356


## 7. Sandbox — write your own queries

Change the SQL below and run the cell to explore anything you want.

In [ ]:
# Your playground — edit this query and run!
con.execute("""
    SELECT *
    FROM fact_daily_prices
    WHERE ticker = 'AAPL'
    ORDER BY date DESC
    LIMIT 10
""").fetchdf()